# Project Introduction
The project is to build a FAQ chatbot, that will help prospective students to get accurate answers about the LLM-Zoomcamp free course. The system will use as knowledge base the complete set of lessons hosted on Github repository online. Each lesson is a single markdown file.  

I will first create a "static RAG" system and then I will conver it into an "agentic RAG".
## Environment setup
For this project I will use the following setup:  
- `uv` a ultra-fast Python package manager
    - `uv init`
- The following python packages:
    - `gitsource` to retrieve the knowledge base from GitHub
    - `minsearch` to create an "in-memory" search engine for the knowledge base
    - `openai` to communicate with the LLM model
    - `jupyter` to work in a jupyter notebook
    - `python-dotenv` to load environmental variables

### Phase 1: Static RAG

In the Phase 1, I will build the static version of this RAG System.
As the acronym suggest the main architecture will follow three major steps:

1. Retrieval of all course's lesson from the GitHub repository
2. Augmentation of the prompt with ***instructions***, ***context***, and the ***user query***
3. Generation of an output by the LLM to provide the user with an answer that is as accurate as possible  
    (we will see the limits of the static RAG mechanics)

#### 1.1 Retrieval (fetch knowledge base)

In [47]:
# Retrieve all the course's lessons from the GitHub repository
from gitsource import GithubRepositoryDataReader

# Instatiate a GithubRepositoryDataReader object to read the course's lessons
repo_reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"}, # Use a set and not a list as it is collection of unique values and faster to lookup
    filename_filter=lambda path: "/lessons" in path,
)

# Read in the course's lessons
# <- list(RawRepositoryFile)
lesson_files = repo_reader.read()

In [48]:
# Find how many total lessons are in the LLM-Zoomcamp course
print(f"Found {len(lesson_files)} lessons")

Found 72 lessons


In [49]:
# Find what standard methods and attributes the RawRepositoryFile object has
methods = [method for method in dir(lesson_files[0]) if not method.startswith("_")]
print(methods)

['content', 'filename', 'parse']


In [50]:
# List to store each lesson file parsed into a dict
lesson_docs = []

# The method parse will parse each lesson RawRepositoryFile object into a dict
for lesson in lesson_files:
    lesson_docs.append(lesson.parse())

In [51]:
# Inspect first 3 parsed lessons
lesson_docs[:3]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

#### 1.2 Retrieval (make the knowledge base - `lesson_files`, searchable)

In [52]:
import minsearch

A `minsearch` implementation requires exactly three steps:

1. Creating a `minsearch` `Index` and declare as arguments what fields of our data we want to make ***searchable*** (`text_fields`) and which field will act as a filter to exclude entire documents from a search (`keyword_fields`).
2. Fitting the `Index`: We feed our full knowledge base to `minsearch` to create the *inverted index* to make our knwoledge base searchable and filterable.
3. Perform a search: We pass a question (`user_query`) to get back a ranked list of matching documents from our knowledge base.

In [53]:
# Print the keys of one obejct in our knowledge base to see what can be used as text_field and keyword_field in our minsearch implementation
list(lesson_docs[0])

['content', 'filename']

In [54]:
# Instatiate a minsearch Index object
lesson_index = minsearch.Index(
    text_fields=["content"],
    # This parameter is only needed if we use filter_dict in the search method
    keyword_fields=["filename"]
)

# Create the actual searchable index of the knowledge base (lesson_files)
lesson_index.fit(lesson_docs)

##### Making a first search in our knowledge base

In [55]:
user_query = "How does the agentic loop keep calling the model until it stops?"

# I only pass the user_query, as boost_dict is not needed as we only one text_fields
# Also, I won't pass the filter_dict parameter as we don't need to exclude any lesson from the search
# In this search we only keep the first result
lesson_index.search(user_query, num_results=1)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [56]:
search_results = lesson_index.search(user_query)

# Inspect first two results
search_results[:2]

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

#### Using the RAGPipeline Class

In [57]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

True

In [58]:
from importlib import reload
import rag_schema

reload(rag_schema)
from rag_schema import RAGPipeline

my_rag_pipeline = RAGPipeline(
    index_base=lesson_index,
    llm_client=OpenAI()
)

In [64]:
answer = my_rag_pipeline.execute_rag("How does the agentic lowop keep calling the model until it stops?")

In [65]:
print(answer)

('The agentic loop repeatedly calls the model inside a `while True` loop and checks whether the model returned any `function_call` items.\n\nHow it works:\n1. Send the current `messages` history to the model.\n2. If the model returns a `function_call`, run the tool and append the tool result to `messages`.\n3. If the model returns a normal `message` with no function calls, set `has_function_calls = False`.\n4. Stop the loop when there are no function calls in that turn.\n\nSo the exit condition is: **no function calls in the latest response means the agent is done**.', ResponseUsage(input_tokens=6658, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=132, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=6790))
